# 03 · Gold marts and change data capture

Run after the Phase 5 steps in the README. Same kernel setup as notebook 00.

You'll see: the three Gold marts answering business questions, how an order change travelled
Kafka → Bronze → Silver (MERGE) → Gold (incremental), and the watermark that makes it work.

In [ ]:
import sys
sys.path.insert(0, '/home/jovyan/src')

from delta.tables import DeltaTable
from pyspark.sql import functions as F
from common.spark_session import get_spark, lake_path

spark = get_spark('gold-check')
gold = lambda t: spark.read.format('delta').load(lake_path('gold', t))

## 1. Monthly revenue (from `sales_daily`)

In [ ]:
(gold('sales_daily')
   .groupBy(F.trunc('order_purchase_date', 'month').alias('month'))
   .agg(F.sum('revenue').alias('revenue'), F.sum('orders').alias('orders'))
   .orderBy('month').show(30))

## 2. Which states wait longest, and does it hurt reviews? (`delivery_by_state`, 2018)

In [ ]:
(gold('delivery_by_state').filter(F.year('purchase_month') == 2018)
   .groupBy('customer_state')
   .agg(F.sum('delivered_orders').alias('orders'),
        F.round(F.sum(F.col('avg_delivery_days') * F.col('delivered_orders')) / F.sum('delivered_orders'), 1).alias('avg_days'),
        F.round(F.sum('late_orders') / F.sum('delivered_orders'), 3).alias('late_rate'),
        F.round(F.sum(F.col('avg_review_score') * F.col('delivered_orders')) / F.sum('delivered_orders'), 2).alias('avg_review'))
   .orderBy(F.desc('avg_days')).show(10))

## 3. Seller scorecard: high-volume sellers with the worst reviews

In [ ]:
(gold('seller_scorecard').filter('orders >= 100')
   .orderBy('avg_review_score').show(10, truncate=False))

## 4. Follow one change through the pipeline

`order_updates.py` marked some shipped orders as delivered. Find one in Bronze (two events for the same
order), then in Silver's Change Data Feed (the MERGE recorded a before and after image).

In [ ]:
bronze_orders = spark.read.format('delta').load(lake_path('bronze', 'orders'))
updated = bronze_orders.filter(F.col('_source') == 'order_update_simulator')
example = updated.select('order_id').first()[0]
print('Example order:', example)

(bronze_orders.filter(F.col('order_id') == example)
   .select('order_status', 'order_delivered_customer_date', '_source', '_kafka_timestamp')
   .orderBy('_kafka_timestamp').show(truncate=False))

In [ ]:
silver_orders = lake_path('silver', 'orders')
# Change Data Feed only exists from the commit that switched it on (build_silver.py does that)
cdf_from = (DeltaTable.forPath(spark, silver_orders).history()
              .filter("operation = 'SET TBLPROPERTIES'").agg(F.max('version')).first()[0])

(spark.read.format('delta').option('readChangeFeed', 'true').option('startingVersion', cdf_from)
   .load(silver_orders)
   .filter(F.col('order_id') == example)
   .select('_change_type', '_commit_version', 'order_status', 'order_delivered_customer_date', 'last_event_at')
   .orderBy('_commit_version', '_change_type').show(truncate=False))

## 5. The MERGE history of `silver.orders`, and Gold's watermark

In [ ]:
(DeltaTable.forPath(spark, silver_orders).history()
   .select('version', 'operation',
           F.col('operationMetrics.numTargetRowsUpdated').alias('updated'),
           F.col('operationMetrics.numTargetRowsInserted').alias('inserted'))
   .show(10))

spark.read.format('delta').load(lake_path('_state', 'gold_watermark')).show()

## All green? Phase 5 is done.

Next: Phase 6, the same Gold logic as dbt models and tests in Snowflake.